# Spam Detection Model Evaluation
**Machine Learning · Classification**

---

## Background

A first-generation spam filter has been deployed, but the team has raised concerns: important 
messages are occasionally misclassified, and confidence in the model's reliability is limited. 
Before moving to production at scale, we need a rigorous evaluation.

This notebook answers the core question: **which classification algorithm should the team 
trust to power the spam filter, and why?**

---

## What this notebook covers

| Step | Description |
|------|-------------|
| **1 — Data preparation** | Load, inspect, and clean `messages.csv`. Handle missing values and standardize labels. |
| **2 — Feature engineering** | Convert raw text into numeric features using TF-IDF. Split data 80/20 with stratification. |
| **3 — Model training** | Train four classifiers: Logistic Regression, Naive Bayes, Decision Tree, Linear SVC. |
| **4 — Evaluation** | Measure accuracy, precision, recall, and F1. Visualize confusion matrices per model. |
| **5 — Conclusion** | Compare all models side by side. Select and justify the best algorithm for the team. |

---

## Why F1 and not just accuracy?

Raw accuracy is misleading on imbalanced datasets — a model that labels everything as "ham" 
would score high but miss all spam. **F1 score balances precision and recall**, making it the 
primary metric for this evaluation. Both false negatives (spam in the inbox) and false positives 
(legitimate messages blocked) carry real cost for the team.

## 📦 Step 1 — Import libraries

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 📊 Step 2 — Load & clean the dataset

### --- Statistics after cleaning --- ---

#### Total rows: 5404

category:
- ham     4682
- spam     722
Name: count, dtype: int64

Average message length:

category
- ham      71.462623
- spam    138.246537
Name: message_len, dtype: float64

In [ ]:
dataset = pd.read_csv("data/messages.csv", encoding="latin-1")
dataset.head(10)

# Checking the dataset - nan values
empty_values = dataset.isnull().sum()
print("Number of NaN values in each column:") 
print(empty_values)

# Checking the column 'category'
category_values = dataset['category'].unique()
print("Unique values in the 'category' column:")
print(category_values)

# Remove rows with missing values in 'message' and 'category' columns
data = dataset.dropna(subset=['message', 'category'])

# Standardize categories
# Convert to lowercase, remove whitespace, and replace 'not spam' with 'ham'
data['category'] = data['category'].str.strip().str.lower()
data['category'] = data['category'].replace('not spam', 'ham')

# Keep only rows that are now labeled as 'ham' or 'spam'
data = data[data['category'].isin(['ham', 'spam'])]

# Add a new column to the table that calculates the character count for each message (Post-cleaning)
data['message_len'] = data['message'].apply(len)

print("--- Statistics after cleaning ---")
print(f"Total rows: {data.shape[0]}")
print(data['category'].value_counts())
print("\nAverage message length:")
print(data.groupby('category')['message_len'].mean())

# Saving cleaned dataset as a new file
data.to_csv("data/messages_cleaned.csv", index=False)
data.head(10)

## ✂️ Step 3 — Split data & vectorize text

We load the cleaned dataset and prepare it for model training:
- **X** — input features (the message text)
- **y** — target labels (`spam` or `ham`)

The data is split **80% training / 20% testing**, with `stratify=y` to preserve 
the spam/ham ratio in both sets.

Text is then converted to numeric form using **TF-IDF vectorization** — each word 
gets a score reflecting how important it is across all messages.

In [ ]:
# Load cleaned dataset
data = pd.read_csv("data/messages_cleaned.csv")

# Define features and target
X = data['message']
y = data['category']

# Split into train and test sets (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Training set size:  {X_train_tfidf.shape[0]} messages")
print(f"Test set size:      {X_test_tfidf.shape[0]} messages")
print(f"Vocabulary size:    {X_train_tfidf.shape[1]} words")